---
authors:
  - edesz
date: 2025-10-04
---

# Get Best Features and Model from Validation

In [ ]:
import pandas as pd
from metaflow import Flow, metadata

In [ ]:
import cc_churn.tuning as tn

## About

In this notebook, we will evaluate all five ML experiments performed as part of the validation analysis performed in the previous notebook.

We will use this evaluation to determine the best selection of features, feature preprocessing and classifier.

### Outputs

None

## User Inputs

In [ ]:
primary_metric_val = "prauc"
threshold_overfit = 5

cols_expt_compare = [
    "experiment_num",
    "run_id",
    "feat_group",
    "model_name",
    "threshold",
    "fit_time",
    "score_time",
    f"train_{primary_metric_val}",
    f"test_{primary_metric_val}",
    "pct_diff",
    "is_overfit",
    "is_overfit_significant",
]

## Comparison of Validation Experiments

The results produced by the experiments to optimize **feature selection** and **model selection** are now compared.

Get a list of all Metaflow flow runs

In [ ]:
metaflow_experiment_runs = list(Flow("ValidationFlow").runs())

Verify all runs are finished

In [ ]:
assert all(list(map(lambda x: x.finished, metaflow_experiment_runs)))

Results from all experiments are combined and sorted in descending order of the primary evaluation metric for model validation (`prauc`) 

In [ ]:
%%time
cv_results_tuned_models = {
    f"{run.data.df_cv['model_name'].head(1).squeeze()}_{run.id}": (
        run.data.df_cv.assign(
            run_id=run.id,
            experiment_num=lambda df: (
                df['feat_group'].map(
                    {
                        "[numericals_1]": 1,
                        "[numericals_2]": 2,
                        "[numericals_1,ordinals]": 3,
                        "[numericals_1,ordinals,categoricals_ohe_encoding]": 4,
                        "[numericals_1,ordinals,categoricals_no_encoding]": 5,
                    }
                )
            )
        )
    )
    for k, run in enumerate(metaflow_experiment_runs, 1)
}
df_cv_scores_outer = tn.combine_cv_scores_thresholds(cv_results_tuned_models)
df_cv_scores_all = tn.agg_cv_scores_thresholds(
    df_cv_scores_outer, primary_metric_val, threshold_overfit
)

### Numericals (Two Groupings)

Results from all six runs of experiments 1 and 2, which compared the choice of two groupings of numerical features, are shown below

In [ ]:
df_cv_scores_all[cols_expt_compare].query(
    "experiment_num.isin([1,2])"
).sort_values(
    by=["model_name", "experiment_num"], ignore_index=True
).style.set_properties(
    subset=["model_name"] + [f"test_{primary_metric_val}"],
    **{"background-color": "yellow", "color": "black"},
)

**Observations**

1. The first grouping of numerical features outperforms the second grouping by a small margin for both the tree-based and linear models. Model performance is assessed using the primary scoring metric for model validation discussed in the project scope, namely `prauc`. Here, we are using this metric calculated on the data that was not used in model training, so the column with the score to consider is `test_prauc`. The model training time was nearly unchanged across both feature groupings.
2. The two best performing classifiers were `XGBClassifier` (second best) and `HistGradientBoostingClassifier` (best).

### Numericals, Numericals+Ordinals, Numericals+Ordinals+OHE Categoricals

Results from all runs of the following experiments are shown below

1. experiment 1 - only using numerical features
2. experiment 3 - using numerical and ordinal features
3. experiment 4 - using all features, including categoricals with one-hot encoding

In [ ]:
df_cv_scores_all[cols_expt_compare].query(
    "experiment_num.isin([1,3,4])"
).sort_values(
    by=["model_name", "experiment_num"], ignore_index=True
).style.set_properties(
    subset=["model_name"] + [f"test_{primary_metric_val}"],
    **{"background-color": "yellow", "color": "black"},
)

**Observations**

1. The inclusion of ordinal and/or one-hot encoded categorical features (experiments 3 and 4) gave a negligible improvement to pipeline performance (shown in `test_prauc`) relative to performance using numerical features only (experiment 1). They did consistently increase the training time (see `fit_time`). The minor improvement was seen in the two best performing tree-based models (`XGBClassifier` and `HistGradientBoostingClassifier`).
2. The `Ensemble__VotingClassifier` has a lower score when it used a linear model (`LogisticRegression`) instead of one of the two best performing tree-based classifiers (`XGBClassifier` or `HistGradientBoostingClassifier`).

### Numericals, Numericals+Ordinals, Numericals+Ordinals+Unencoded Categoricals

Results from all runs of the following experiments are shown below

1. experiment 1 - only using numerical features
2. experiment 3 - using numerical and ordinal features
3. experiment 5 - using all features, including categoricals with categorical encoding performed by the classifier

In [ ]:
df_cv_scores_all[cols_expt_compare].query(
    "experiment_num.isin([1,3,5])"
).query("model_name == 'HistGradientBoostingClassifier'").sort_values(
    by=["model_name", "experiment_num"], ignore_index=True
).style.set_properties(
    subset=["model_name"] + [f"test_{primary_metric_val}"],
    **{"background-color": "yellow", "color": "black"},
)

**Observations**

1. Again, the inclusion of ordinal and/or unencoded categorical features (experiments 3 and 5) did not improve pipeline performance (`test_prauc`) relative to performance using numerical features only (experiment 1).
As with the previous observations, `XGBClassifier` (second best) and `HistGradientBoostingClassifier` (best) were the top two performing classifiers in terms of the primary evaluation metric (`prauc`) on the test split (`test_prauc`).

### General Observations

1. In all experiments, tree-based models out-perform the linear models.
2. In all the experiments in which it was used, `HistGradientBoostingClassifier` slightly outperformed `XGBClassifier` and `Ensemble__VotingClassifier`, and strongly outperformed `RandomForestClassifier`. The added disadvantage of `Ensemble__VotingClassifier` is that its training times take longer than other two.
3. In all experiments with top-performing tree-based models (`Ensemble__VotingClassifier`, `XGBClassifier`, `HistGradientBoostingClassifier`), overfitting (in the `pct_diff` column) falls in the 5-10% range. For linear models and `RandomForestClassifier`, overfitting is less than 5%.
4. As seen from comparing experiments 1 or 2 (which did not include sensitive features) to 3, 4 or 5 (which did include sensitive features), model performance was not negatively impacted if sensitive features were excluded.
6. Experiments 3, 4 and 5 verified the hypothesis we formed in the EDA notebook that only numerical features are sufficient to predict credit card customer churn since the inclusion of ordinals and categoricals did not improve model validation scores.

### Main Findings

Based on the combination of the primary evaluation metric and training time across all five experiments, the best combination of pipeline steps (features, pre-processor and ML classifier) is to use only the first grouping of numerical features that are min-max scaled (in experiment 1) and the `HistGradientBoostingClassifier` classifier model.

All sensitive features can be excluded from the training data without negatively impacting model performance. This eliminates the possibliity of discrimination related to these attributes.

### Get Metadata for Best Experiment

Based on the mean outer CV scores, get the best

1. experiment run (`run_id`)
2. experiment number
3. classifier name
4. average decision threshold

In [ ]:
# get best experiment run
df_cv_scores_best = df_cv_scores_all.query("(experiment_num == 1)").nlargest(
    1, [f"test_{primary_metric_val}"]
)

# get best Metaflow run ID
best_experiment_run_id = df_cv_scores_best["run_id"].squeeze()

# get best experiment number
best_experiment_num = df_cv_scores_best["experiment_num"].squeeze()

# get best classifier
best_model_name = df_cv_scores_best["model_name"].squeeze()

# get best decision threshold
best_estimator_threshold = df_cv_scores_best["threshold"].squeeze()
print(best_experiment_run_id)

#### Estimate of Model Reliability

The standard deviation of the prediction scores across outer CV folds indicates a model's stability within each experiment. A high standard deviation suggests the model is sensitive to specific training data subsets, potentially indicating high variance or overfitting.

We will estimate model reliability from the outer CV scores during model validation by calculating the [coefficient of variation (COV)](https://en.wikipedia.org/wiki/Coefficient_of_variation), which is the ratio of standard deviation to the mean, of the primary validation metric for all experiment runs

In [ ]:
%%time
df_cov = (
    pd.concat(
        [
            (
                df_cv.groupby(
                    ["feat_group", "experiment_num", "run_id", "model_name"]
                )
                .agg(
                    {
                        f"train_{primary_metric_val}": ["mean", "std"],
                        f"test_{primary_metric_val}": ["mean", "std"],
                    }
                )
                .set_axis(
                    [
                        f"train_mean_{primary_metric_val}",
                        f"train_std_{primary_metric_val}",
                        f'test_mean_{primary_metric_val}',
                        f'test_std_{primary_metric_val}',
                    ],
                    axis=1,
                )
                .reset_index()
            )
            for k, df_cv in cv_results_tuned_models.items()
        ]
    )
    .sort_values(by=["experiment_num"], ascending=True, ignore_index=True)
    .assign(
        train_cov=lambda df: (
            df[f"train_std_{primary_metric_val}"]
            .div(df[f"train_mean_{primary_metric_val}"])
            .mul(100)
        ),
        test_cov=lambda df: (
            df[f"test_std_{primary_metric_val}"]
            .div(df[f"test_mean_{primary_metric_val}"])
            .mul(100)
        ),
    )
)
df_cov

**Observations**

1. Across all outer CV folds for all classifiers (models) and across all experiment runs, the COV of the primary validation metric (`prauc`) is at most 5.1% on the validation split of each CV fold. This is higher than the COV of the training split per CV fold, which is less than 2%. This is partially explained by the relatively smaller size of the validation fold (~80:20 split per CV fold).
2. For the best classifier (`HistGradientBoostingClassifier`) and the best experiment run, the COV is an order of magnitude higher in the validation fold than in the train fold. In fact, for this model, the COV is consistently between 5-10 times higher on the validation fold than the train fold. Notably, when more types of features are included, the COV reduces from ~10X (experiment 1) to ~2.5X (experiment 5). It is reassuring that COVs are relatively small (less than ~2.2%). So, we will assume the larger value on unseen (validation) data is primarily due to its relatively smaller size and that this does not indicate an underlying difference in the unseen data relative to the training data.

## Limitations

1. Hyperparameter optimization was not performed.
2. Not all models (classifiers) that support un-encoded categorical features were compared (eg. `XGBClassifier`).

## Conclusion

The inclusion of ordinal or categorical features made a negligible impact on model performance using `prauc`. This validates the hypothesis we formed during EDA that numerical features were sufficient to predict credit card customer churn.

Tree-based models have outperformed linear models. In the EDA notebook, we saw the need for using tree-based models from the skewed disributions of the numerical features that were used in model training. This finding about the relative outperformance by tree-based models verifies that observation.

Based on model performance, the validation analysis here indicates

1. the best features are numerical features only
2. that features hould be preprocessed using `scikit-learn`'s `MinMaxScaler()`
3. the best model (classifier) is `scikit-learn`'s `HistGradientBoostingClassifier`